In [1]:
# =========================
# Cell 0: imports & paths
# =========================
import os
import re
import gc
import json
from pathlib import Path
from collections import defaultdict

import torch
from PIL import Image
from IPython.display import display
from tqdm import tqdm

from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

DATA_DIR = Path("/scratch/zdong112/krr-data")
TEST_JSON = DATA_DIR / "test.json"
ANSWER_JSON = DATA_DIR / "test_answer.json"
IMAGE_DIR = DATA_DIR / "image"
GRAPH_DIR = DATA_DIR / "final_vlm_kr.json"
OUTPUT_DIR = DATA_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("TEST_JSON:", TEST_JSON)
print("ANSWER_JSON:", ANSWER_JSON)
print("IMAGE_DIR:", IMAGE_DIR)
print("GRAPH_DIR:", GRAPH_DIR)

/etc/python/sitecustomize.py:117: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  mod = _original_import(name, globals, locals, fromlist, level)


DATA_DIR: /scratch/zdong112/krr-data
TEST_JSON: /scratch/zdong112/krr-data/test.json
ANSWER_JSON: /scratch/zdong112/krr-data/test_answer.json
IMAGE_DIR: /scratch/zdong112/krr-data/image
GRAPH_DIR: /scratch/zdong112/krr-data/final_vlm_kr.json


In [2]:
# =========================
# Cell 1: load model
# =========================
model_name = "Qwen/Qwen2.5-VL-7B-Instruct"

processor = AutoProcessor.from_pretrained(model_name)
processor.tokenizer.padding_side = "left"

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()
print("model loaded")

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 729/729 [00:02<00:00, 323.27it/s]


model loaded


In [3]:
# =========================
# Cell 2: load data
# =========================
with open(TEST_JSON, "r") as f:
    test_data = json.load(f)

with open(ANSWER_JSON, "r") as f:
    test_answer = json.load(f)

print("num test samples:", len(test_data))
print("num answer samples:", len(test_answer))

print("\nexample test sample:")
print(test_data[0])

print("\nexample answer sample:")
print(test_answer[0])

num test samples: 10002
num answer samples: 10002

example test sample:
{'scene': 'black background', 'object': ['glass', 'rubber bullet'], 'source': 'web', 'file_name': ['iNINChj51Aqn.mp4', 'iNINChj51Aqj.png', 'iNINChj51Aqk.png', 'iNINChj51Aql.png', 'iNINChj51Aqm.png'], 'description': None, 'question': "Following the content of the <video>, which option's corresponding picture will happen first?\nA. <image>\nB. <image>\nC. <image>\nD. <image>\n", 'mode': 'general', 'idx': 0, 'split': 'val'}

example answer sample:
{'idx': 0, 'answer': 'A', 'task_type': 'dynamics', 'sub_type': 'collision', 'ability_type': 'prediction', 'mode': 'general'}


In [4]:
# =========================
# Cell 3: answer map
# =========================
answer_map = {item["idx"]: item["answer"] for item in test_answer}

print("answer_map size:", len(answer_map))
print("answer for idx=0:", answer_map[0])

answer_map size: 10002
answer for idx=0: A


In [5]:
# =========================
# Cell 4: load KR
# =========================
with open(GRAPH_DIR, "r") as f:
    kg_merged = json.load(f)

print(type(kg_merged), len(kg_merged))
print(kg_merged[0])

<class 'list'> 10002
{'idx': 0, 'question': "Following the content of the <video>, which option's corresponding picture will happen first?\nA. <image>\nB. <image>\nC. <image>\nD. <image>\n", 'entities': ['video'], 'kr': [['pressure', 'increases', 'temperature'], ['increased temperature', 'leads to', 'expansion'], ['spread', 'fragments', 'final state'], ['breaking', 'produces', 'fragments'], ['smoke', 'causes', 'pressure'], ['fragments', 'spread', 'around'], ['glass', 'has', 'smoke'], ['expansion', 'results in', 'breaking'], ['video', 'contains', 'image'], ['image', 'shows', 'glass']], 'raw': '(video, contains, image)\n\n(image, shows, glass)\n\n(glass, has, smoke)\n\n(smoke, causes, pressure)\n\n(pressure, increases, temperature)\n\n(increased temperature, leads to, expansion)\n\n(expansion, results in, breaking)\n\n(breaking, produces, fragments)\n\n(fragments, spread, around)\n\n(spread, fragments, final state)'}


In [6]:
# =========================
# Cell 5: graph subset
# =========================
graph_keys = set(str(x["idx"]) for x in kg_merged)

graph_subset = [
    sample for sample in test_data
    if str(sample["idx"]) in graph_keys
]

print("full test size:", len(test_data))
print("graph subset size:", len(graph_subset))
print("example graph sample idx:", graph_subset[0]["idx"])

full test size: 10002
graph subset size: 10002
example graph sample idx: 0


In [7]:
# =========================
# Cell 6: image loading
# =========================
def build_image_paths(sample):
    files = sample.get("file_name", [])

    if isinstance(files, str):
        files = [files]

    if not isinstance(files, list):
        return []

    image_files = [
        f for f in files
        if isinstance(f, str) and f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
    ]

    return [IMAGE_DIR / f for f in image_files]


def load_images(sample):
    paths = build_image_paths(sample)
    images = []

    for p in paths:
        if p.exists():
            images.append(Image.open(p).convert("RGB"))
        else:
            print("missing:", p)

    return images

In [8]:
# =========================
# Cell 7: knowledge getter
# =========================
def get_knowledge(graph, sample):
    if graph is None:
        return {}

    idx = sample["idx"]

    # 同时兼容 int key 和 str key
    if idx in graph:
        return graph[idx]
    if str(idx) in graph:
        return graph[str(idx)]

    return {}


def filter_triples(triples):
    return triples

In [9]:
# =========================
# Cell 8: KR -> text
# =========================
def knowledge_to_text(knowledge, max_kr=8):
    """
    适配你现在的 KR 格式:
    {
        "idx": ...,
        "kr": [
            ["pressure", "increases", "temperature"],
            ...
        ],
        "raw": ...
    }

    策略:
    1. 优先使用 knowledge["kr"]
    2. 过滤过于视觉描述/模板化的关系
    3. 因果类关系优先
    4. 最多保留 max_kr 条，避免 prompt 太长和 attention dilution
    """
    if not knowledge:
        return ""

    if isinstance(knowledge, dict):
        triples = knowledge.get("kr", [])
        if not triples:
            return ""
    elif isinstance(knowledge, list):
        triples = knowledge
    else:
        return str(knowledge)

    triples = [
        t for t in triples
        if isinstance(t, (list, tuple)) and len(t) == 3
    ]

    if not triples:
        return ""

    def is_useful(t):
        text = " ".join(map(str, t)).lower()

        # 这些通常是弱视觉描述，不太帮助物理推理
        useless_words = [
            "contains",
            "shows",
            "has",
            "depicts",
            "includes",
            "is shown",
            "appears"
        ]

        return not any(w in text for w in useless_words)

    triples = [t for t in triples if is_useful(t)]

    if not triples:
        return ""

    def is_causal(t):
        rel = str(t[1]).lower()
        causal_keywords = [
            "cause",
            "causes",
            "lead",
            "leads",
            "result",
            "results",
            "produce",
            "produces",
            "increase",
            "increases",
            "decrease",
            "decreases",
            "trigger",
            "triggers",
            "affect",
            "affects",
            "generate",
            "generates",
            "break",
            "breaks"
        ]
        return any(k in rel for k in causal_keywords)

    causal = [t for t in triples if is_causal(t)]
    others = [t for t in triples if not is_causal(t)]

    selected = (causal + others)[:max_kr]

    lines = []
    for h, r, t in selected:
        lines.append(f"{h} --{r}--> {t}")

    return "\n".join(lines)

In [10]:
# =========================
# Cell 9: quick KR check
# =========================
sample = kg_merged[0]

print("RAW KR:")
print(sample.get("kr", []))

print("\nKNOWLEDGE TEXT:")
text = knowledge_to_text(sample, max_kr=8)
print(text)

RAW KR:
[['pressure', 'increases', 'temperature'], ['increased temperature', 'leads to', 'expansion'], ['spread', 'fragments', 'final state'], ['breaking', 'produces', 'fragments'], ['smoke', 'causes', 'pressure'], ['fragments', 'spread', 'around'], ['glass', 'has', 'smoke'], ['expansion', 'results in', 'breaking'], ['video', 'contains', 'image'], ['image', 'shows', 'glass']]

KNOWLEDGE TEXT:
pressure --increases--> temperature
increased temperature --leads to--> expansion
breaking --produces--> fragments
smoke --causes--> pressure
expansion --results in--> breaking
spread --fragments--> final state
fragments --spread--> around


In [11]:
# =========================
# Cell 10: inspect sample
# =========================
def inspect_sample(sample, graph):
    gold = answer_map[sample["idx"]]

    print("=" * 100)
    print("IDX:", sample["idx"])
    print("GOLD:", gold)

    print("\nQUESTION:")
    print(sample["question"])

    print("\nFILE_NAME:")
    print(sample.get("file_name", None))

    print("\nIMAGES:")
    imgs = load_images(sample)
    print("num_images:", len(imgs))

    for i, img in enumerate(imgs):
        print(f"image {i}")
        display(img)

    knowledge = get_knowledge(graph, sample)

    print("\nRAW KNOWLEDGE KEYS:")
    if isinstance(knowledge, dict):
        print(list(knowledge.keys()))
    else:
        print(type(knowledge))

    if isinstance(knowledge, dict):
        print("\nKR:")
        for t in knowledge.get("kr", [])[:20]:
            print(t)

        if "raw" in knowledge:
            print("\nRAW:")
            print(knowledge.get("raw", "")[:1000])

    print("\nFILTERED KNOWLEDGE TEXT:")
    print(knowledge_to_text(knowledge, max_kr=8))

In [17]:
kg_map = {x["idx"]: x for x in kg_merged}
inspect_sample(sample, kg_map)

IDX: 0
GOLD: A

QUESTION:
Following the content of the <video>, which option's corresponding picture will happen first?
A. <image>
B. <image>
C. <image>
D. <image>


FILE_NAME:
None

IMAGES:
num_images: 0

RAW KNOWLEDGE KEYS:
['idx', 'question', 'entities', 'kr', 'raw']

KR:
['pressure', 'increases', 'temperature']
['increased temperature', 'leads to', 'expansion']
['spread', 'fragments', 'final state']
['breaking', 'produces', 'fragments']
['smoke', 'causes', 'pressure']
['fragments', 'spread', 'around']
['glass', 'has', 'smoke']
['expansion', 'results in', 'breaking']
['video', 'contains', 'image']
['image', 'shows', 'glass']

RAW:
(video, contains, image)

(image, shows, glass)

(glass, has, smoke)

(smoke, causes, pressure)

(pressure, increases, temperature)

(increased temperature, leads to, expansion)

(expansion, results in, breaking)

(breaking, produces, fragments)

(fragments, spread, around)

(spread, fragments, final state)

FILTERED KNOWLEDGE TEXT:
pressure --increases-->

In [15]:
# =========================
# Cell 11: baseline prompt
# =========================
def build_baseline_prompt(sample):
    num_images = len(build_image_paths(sample))
    question = sample["question"]

    if num_images == 0:
        return f"""Answer the following multiple-choice question.

Question:
{question}

Respond with ONLY one letter: A, B, C, or D.
Do not explain your answer."""

    elif num_images == 1:
        return f"""Answer the question based on the image.

Question:
{question}

Respond with ONLY one letter: A, B, C, or D.
Do not explain your answer."""

    elif num_images == 4:
        return f"""You are given candidate images for a multiple-choice question.

Question:
{question}

Respond with ONLY one letter: A, B, C, or D.
Do not explain your answer."""

    else:
        return f"""Answer the question based on the provided images.

Question:
{question}

Respond with ONLY one letter: A, B, C, or D.
Do not explain your answer."""

In [16]:
# =========================
# Cell 12: graph prompt
# =========================
def build_graph_prompt(sample, knowledge_text):
    question = sample["question"]

    return f"""Answer the question based on the images and the provided physical knowledge.

Question:
{question}

Relevant physical knowledge:
{knowledge_text}

Use the knowledge only if it helps answer the question.

Respond with ONLY one letter: A, B, C, or D.
Do not explain your answer."""

In [18]:
# =========================
# Cell 13: build messages
# =========================
def build_messages(sample, prompt):
    img_paths = build_image_paths(sample)

    content = []

    for p in img_paths:
        content.append({
            "type": "image",
            "image": str(p)
        })

    content.append({
        "type": "text",
        "text": prompt
    })

    return [{"role": "user", "content": content}]

In [19]:
# =========================
# Cell 14: cached image loading
# =========================
image_cache = {}

def load_images_cached(sample):
    paths = build_image_paths(sample)
    images = []

    for p in paths:
        key = str(p)

        if key not in image_cache:
            if not p.exists():
                print("missing:", p)
                continue
            image_cache[key] = Image.open(p).convert("RGB")

        images.append(image_cache[key])

    return images

In [20]:
# =========================
# Cell 15: single inference
# =========================
@torch.inference_mode()
def run_single(sample, prompt, max_new_tokens=16):
    images = load_images_cached(sample)
    messages = build_messages(sample, prompt)

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    if len(images) == 0:
        inputs = processor(
            text=[text],
            return_tensors="pt"
        )
    else:
        inputs = processor(
            text=[text],
            images=images,
            return_tensors="pt"
        )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True
    )

    trimmed = [
        o[len(i):] for i, o in zip(inputs["input_ids"], out)
    ]

    answer = processor.batch_decode(
        trimmed,
        skip_special_tokens=True
    )[0]

    del inputs, out, trimmed
    torch.cuda.empty_cache()

    return answer.strip()

In [21]:
# =========================
# Cell 16: batch inference
# =========================
@torch.inference_mode()
def run_batch(samples, prompts, max_new_tokens=16):
    batch_images = [load_images_cached(s) for s in samples]
    batch_messages = [
        build_messages(s, p)
        for s, p in zip(samples, prompts)
    ]

    batch_text = [
        processor.apply_chat_template(
            m,
            tokenize=False,
            add_generation_prompt=True
        )
        for m in batch_messages
    ]

    image_counts = [len(x) for x in batch_images]

    # batch 内图片数量必须一致，否则 processor 容易出错
    if len(set(image_counts)) != 1:
        return [
            run_single(s, p, max_new_tokens=max_new_tokens)
            for s, p in zip(samples, prompts)
        ]

    num_images = image_counts[0]

    # 关键修复:
    # num_images == 0 时不要传 images=batch_images
    if num_images == 0:
        inputs = processor(
            text=batch_text,
            return_tensors="pt",
            padding=True
        )
    else:
        inputs = processor(
            text=batch_text,
            images=batch_images,
            return_tensors="pt",
            padding=True
        )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True
    )

    trimmed = [
        o[len(i):] for i, o in zip(inputs["input_ids"], out)
    ]

    answers = processor.batch_decode(
        trimmed,
        skip_special_tokens=True
    )

    del inputs, out, trimmed
    torch.cuda.empty_cache()

    return [a.strip() for a in answers]

In [22]:
# =========================
# Cell 17: extract choice
# =========================
def extract_choice(text):
    if text is None:
        return ""

    text = str(text).strip().upper()

    # 优先匹配单独的 A/B/C/D
    m = re.search(r"\b([ABCD])\b", text)
    if m:
        return m.group(1)

    # fallback
    m = re.search(r"[ABCD]", text)
    if m:
        return m.group(0)

    return ""

In [23]:
# =========================
# Cell 18: experiment runner
# =========================
def group_by_num_images(data):
    groups = {}

    for sample in data:
        n = len(build_image_paths(sample))
        if n not in groups:
            groups[n] = []
        groups[n].append(sample)

    return groups


def run_experiment_batched(
    data,
    mode="baseline",
    graph=None,
    max_samples=None,
    batch_size=4,
    max_kr=8,
    max_new_tokens=16
):
    if max_samples is not None:
        data = data[:max_samples]

    results = []
    groups = group_by_num_images(data)

    for num_images, samples_group in groups.items():
        print(f"Running group with num_images={num_images}, size={len(samples_group)}")

        # 动态 batch size，避免多图 OOM
        if num_images >= 2:
            cur_batch_size = 1
        elif num_images == 1:
            cur_batch_size = min(batch_size, 2)
        else:
            cur_batch_size = batch_size

        for i in tqdm(
            range(0, len(samples_group), cur_batch_size),
            desc=f"{mode}-img{num_images}"
        ):
            batch_samples = samples_group[i:i + cur_batch_size]

            prompts = []

            for sample in batch_samples:
                if mode == "baseline":
                    prompt = build_baseline_prompt(sample)

                else:
                    knowledge = get_knowledge(graph, sample)
                    knowledge_text = knowledge_to_text(
                        knowledge,
                        max_kr=max_kr
                    )

                    # 如果 KR 为空，不要构建空 knowledge prompt
                    if knowledge_text.strip():
                        prompt = build_graph_prompt(sample, knowledge_text)
                    else:
                        prompt = build_baseline_prompt(sample)

                prompts.append(prompt)

            try:
                preds = run_batch(
                    batch_samples,
                    prompts,
                    max_new_tokens=max_new_tokens
                )

                for sample, prompt, pred in zip(batch_samples, prompts, preds):
                    gold = answer_map[sample["idx"]]

                    results.append({
                        "idx": sample["idx"],
                        "pred": pred,
                        "gold": gold,
                        "pred_choice": extract_choice(pred),
                        "gold_choice": extract_choice(gold),
                        "correct": extract_choice(pred) == extract_choice(gold),
                        "num_images": len(build_image_paths(sample)),
                        "mode": mode,
                        "prompt": prompt
                    })

            except Exception as e:
                print(f"Batch failed, fallback to single. Error: {e}")

                for sample, prompt in zip(batch_samples, prompts):
                    try:
                        pred = run_single(
                            sample,
                            prompt,
                            max_new_tokens=max_new_tokens
                        )

                        gold = answer_map[sample["idx"]]

                        results.append({
                            "idx": sample["idx"],
                            "pred": pred,
                            "gold": gold,
                            "pred_choice": extract_choice(pred),
                            "gold_choice": extract_choice(gold),
                            "correct": extract_choice(pred) == extract_choice(gold),
                            "num_images": len(build_image_paths(sample)),
                            "mode": mode,
                            "prompt": prompt
                        })

                    except Exception as e2:
                        results.append({
                            "idx": sample["idx"],
                            "error": str(e2),
                            "num_images": len(build_image_paths(sample)),
                            "mode": mode,
                            "prompt": prompt
                        })

                torch.cuda.empty_cache()
                gc.collect()

    return results

In [24]:
# =========================
# Cell 19: metrics
# =========================
def compute_acc(results):
    valid = [r for r in results if "correct" in r]
    return sum(r["correct"] for r in valid) / len(valid) if valid else 0.0


def compute_acc_by_num_images(results):
    groups = defaultdict(list)

    for r in results:
        if "correct" in r:
            groups[r["num_images"]].append(r)

    out = {}

    for k, vals in groups.items():
        out[k] = sum(v["correct"] for v in vals) / len(vals)

    return out

In [25]:
# =========================
# Cell 20: run baseline
# =========================
baseline_subset_results = run_experiment_batched(
    graph_subset,
    mode="baseline",
    max_samples=500,
    batch_size=8,
    max_new_tokens=16
)

print("baseline subset acc:", compute_acc(baseline_subset_results))
print("baseline acc by num_images:", compute_acc_by_num_images(baseline_subset_results))

baseline_subset_results[:3]

Running group with num_images=4, size=1


baseline-img4: 100%|██████████| 1/1 [00:03<00:00,  3.83s/it]


Running group with num_images=5, size=1


baseline-img5: 100%|██████████| 1/1 [00:01<00:00,  1.08s/it]


Running group with num_images=1, size=85


...

baseline-img0: 100%|██████████| 49/49 [00:05<00:00,  8.46it/s]


Running group with num_images=2, size=27


baseline-img2: 100%|██████████| 27/27 [00:39<00:00,  1.45s/it]

baseline subset acc: 0.386
baseline acc by num_images: {4: 1.0, 5: 0.0, 1: 0.7411764705882353, 0: 0.31088082901554404, 2: 0.3333333333333333}


[{'idx': 0,
  'pred': 'A',
  'gold': 'A',
  'pred_choice': 'A',
  'gold_choice': 'A',
  'correct': True,
  'num_images': 4,
  'mode': 'baseline',
  'prompt': "You are given candidate images for a multiple-choice question.\n\nQuestion:\nFollowing the content of the <video>, which option's corresponding picture will happen first?\nA. <image>\nB. <image>\nC. <image>\nD. <image>\n\n\nRespond with ONLY one letter: A, B, C, or D.\nDo not explain your answer."},
 {'idx': 1,
  'pred': 'D',
  'gold': 'A',
  'pred_choice': 'D',
  'gold_choice': 'A',
  'correct': False,
  'num_images': 5,
  'mode': 'baseline',
  'prompt': 'Answer the question based on the provided images.\n\nQuestion:\n<image>\nThe light first passes through the convex lens and then the concave lens. Slide the concave lens close to the convex lens. Which of the following options will correspond to the phenomenon in the picture?\nA. <image>\nB. <image>\nC. <image>\nD. <image>\n\n\nRespond with ONLY one letter: A, B, C, or D.\nDo n

In [26]:
# =========================
# Cell 21: run KG
# =========================
kg_map = {x["idx"]: x for x in kg_merged}

merged_results = run_experiment_batched(
    graph_subset,
    mode="kg_merged",
    graph=kg_map,
    max_samples=500,
    batch_size=8,
    max_kr=8,
    max_new_tokens=16
)

print("merged KG acc:", compute_acc(merged_results))
print("merged KG acc by num_images:", compute_acc_by_num_images(merged_results))

merged_results[:3]

Running group with num_images=4, size=1


kg_merged-img4: 100%|██████████| 1/1 [00:02<00:00,  2.07s/it]


Running group with num_images=5, size=1


kg_merged-img5: 100%|██████████| 1/1 [00:00<00:00,  1.74it/s]


Running group with num_images=1, size=85


...

kg_merged-img0: 100%|██████████| 49/49 [00:08<00:00,  5.82it/s]


Running group with num_images=2, size=27


kg_merged-img2: 100%|██████████| 27/27 [00:36<00:00,  1.35s/it]

merged KG acc: 0.562
merged KG acc by num_images: {4: 1.0, 5: 0.0, 1: 0.7529411764705882, 0: 0.5129533678756477, 2: 0.6666666666666666}


[{'idx': 0,
  'pred': 'A',
  'gold': 'A',
  'pred_choice': 'A',
  'gold_choice': 'A',
  'correct': True,
  'num_images': 4,
  'mode': 'kg_merged',
  'prompt': "Answer the question based on the images and the provided physical knowledge.\n\nQuestion:\nFollowing the content of the <video>, which option's corresponding picture will happen first?\nA. <image>\nB. <image>\nC. <image>\nD. <image>\n\n\nRelevant physical knowledge:\npressure --increases--> temperature\nincreased temperature --leads to--> expansion\nbreaking --produces--> fragments\nsmoke --causes--> pressure\nexpansion --results in--> breaking\nspread --fragments--> final state\nfragments --spread--> around\n\nUse the knowledge only if it helps answer the question.\n\nRespond with ONLY one letter: A, B, C, or D.\nDo not explain your answer."},
 {'idx': 1,
  'pred': 'D',
  'gold': 'A',
  'pred_choice': 'D',
  'gold_choice': 'A',
  'correct': False,
  'num_images': 5,
  'mode': 'kg_merged',
  'prompt': 'Answer the question based 